# 1- Importing and matching Data

- Get Parent directory from any location
- Import GEM xml file with original information
- Import firtered Compound Graph

In [1]:
import pandas as pd
import numpy as np
import re
from pathlib import Path
from etc.parse_ids import XMLParser

# Get the notebook's directory and go up to project root
notebook_dir = Path().resolve()
project_root = notebook_dir.parent
data_folder = project_root / "data" / "resources"
Pilot = data_folder / "PilotStudy_All"

# Get graph prevouly fitered
graph_gml = data_folder / "generated" / "modified_graph.gml"
# Get the original graph for matching ids
human1_xml = data_folder / "Human-GEM.xml"

### Consortium sumarized data from different samples
- First we define a method to match the chebis with the HUMAN1
- Then upload data and apply changes

In [2]:
# Auxiliar function to extract the chebi number from a string like "CHEBI:12345"
def extract_single_chebi(value):
    """Extract CHEBI number from a single CHEBI string"""
    if pd.isna(value):
        return None
    match = re.search(r"CHEBI:(\d+)", str(value))
    if match:
        return int(match.group(1))
    return None

- Load the Pilot Study compilation

In [3]:
# Load the data file as a pandas DataFrame
data_file = pd.read_excel(Pilot / "Daniel_Suplementary_info.xlsx", sheet_name="Sheet1")
print(f'Total annotated {data_file.CHEBI_ID_Step2.unique().shape[0]} metabolites')

Total annotated 358 metabolites


- Filter level 1 ids

In [4]:
# Filter to only include rows where ID_level is 1
data_file = data_file[data_file.ID_level==1]
print(f'Total annotated {data_file.CHEBI_ID_Step2.unique().shape[0]} metabolites level 1')

Total annotated 228 metabolites level 1


- Add a chebi number only column

In [5]:
# Add new column with extracted CHEBI numbers
data_file['CHEBI_number'] = data_file.CHEBI_ID_Step2.apply(extract_single_chebi)

### From HUMAN1 GEM xml file 
- Create a dataframe
- Create colum to match chebis ids and link them with human1 ids

In [6]:
parser = XMLParser(human1_xml)
df = parser.extract_data()
df_human1 = parser.to_identifier_df()
df_human1['Consortium'] = None

### Matching and filling column

In [7]:
# Convert df_human1 CHEBI to integers for matching
df_human1['chebi_int'] = pd.to_numeric(df_human1['chebi'], errors='coerce').astype('Int64')
# Get set of CHEBIs from data_file
data_file_chebis = set(data_file['CHEBI_number'].dropna().values)

# Fill Consortium column with boolean: True if CHEBI matches, False otherwise
df_human1['Consortium'] = df_human1['chebi_int'].isin(data_file_chebis)

print(f"Consortium column filled with boolean values")
print(f"True matches: {df_human1[df_human1['Consortium'] == True].chebi.unique().shape[0]}")
print(f"False (no match): {(~df_human1['Consortium']).sum()}")


Consortium column filled with boolean values
True matches: 105
False (no match): 8100


In [8]:
# Find CHEBIs in data_file that are NOT in df_human1
df_human1_chebis = set(df_human1['chebi_int'].dropna().values)
data_file_chebis_set = set(data_file['CHEBI_number'].dropna().values)

# CHEBIs not found in df_human1
unmatched_chebis = data_file_chebis_set - df_human1_chebis

print(f"Total CHEBIs in data_file: {len(data_file_chebis_set)}")
print(f"CHEBIs found in df_human1: {len(data_file_chebis_set - unmatched_chebis)}")
print(f"CHEBIs NOT found in df_human1: {len(unmatched_chebis)}")
print(f"\nUnmatched CHEBIs: {sorted(unmatched_chebis)}")

Total CHEBIs in data_file: 228
CHEBIs found in df_human1: 105
CHEBIs NOT found in df_human1: 123

Unmatched CHEBIs: [np.int64(4139), np.int64(4828), np.int64(9008), np.int64(11901), np.int64(14314), np.int64(15620), np.int64(15761), np.int64(16020), np.int64(16113), np.int64(16119), np.int64(16231), np.int64(16312), np.int64(16347), np.int64(16373), np.int64(16831), np.int64(17012), np.int64(17016), np.int64(17072), np.int64(17234), np.int64(17597), np.int64(17687), np.int64(17780), np.int64(18123), np.int64(19030), np.int64(19062), np.int64(19065), np.int64(19660), np.int64(21264), np.int64(21553), np.int64(21563), np.int64(21756), np.int64(21949), np.int64(25858), np.int64(27410), np.int64(27596), np.int64(27732), np.int64(27747), np.int64(27838), np.int64(28123), np.int64(28177), np.int64(28238), np.int64(28393), np.int64(28664), np.int64(28717), np.int64(28775), np.int64(28821), np.int64(28871), np.int64(28946), np.int64(30753), np.int64(30776), np.int64(30845), np.int64(35280), np

note: 123 metbolites from the data couldn't be match on HUMAN1, note some are lipis and most of them considered sidecompounds that will not be taken into account in the model. Also we only included level 1 annotated metabolites

### Mask separated annotations on HUMAN1 dataframe
- For different devices
- For different labs

In [9]:
capitainer = data_file[data_file.Capitainer == "Yes"]
mitra = data_file[data_file.Mitra == "Yes"]
blood = data_file[data_file.Blood == "Yes"]
plasma = data_file[data_file.Plasma == "Yes"]

devices = {
    "Capitainer": capitainer,
    "Mitra": mitra,
    "Blood": blood,
    "Plasma": plasma
}
for device_name, device_df in devices.items():
    chebis = set(device_df['CHEBI_number'].dropna().values)
    df_human1[device_name] = df_human1['chebi_int'].isin(device_df['CHEBI_number'].dropna().values)

    print(f"{device_name} column filled with boolean values")
    print(f"True matches: {df_human1[device_name].sum()}")
    print(f"False (no match): {(~df_human1[device_name]).sum()}")

Capitainer column filled with boolean values
True matches: 332
False (no match): 8124
Mitra column filled with boolean values
True matches: 338
False (no match): 8118
Blood column filled with boolean values
True matches: 321
False (no match): 8135
Plasma column filled with boolean values
True matches: 330
False (no match): 8126


In [10]:
hmgu = data_file[data_file.HMGU == 1]
cembio = data_file[data_file.CEMBIO == 1]
afekta = data_file[data_file.AFEKTA == 1]
icl = data_file[data_file.ICL == 1]
auth = data_file[data_file.AUTh == 1]

labs = {
    "HMGU": hmgu,
    "CEMBIO": cembio,
    "AFEKTA": afekta,
    "ICL": icl,
    "AUTh": auth
    }
for lab_name, lab_df in labs.items():
    chebis = set(lab_df['CHEBI_number'].dropna().values)
    df_human1[lab_name] = df_human1['chebi_int'].isin(lab_df['CHEBI_number'].dropna().values)

    print(f"{lab_name} column filled with boolean values")
    print(f"True matches: {df_human1[lab_name].sum()}")
    print(f"False (no match): {(~df_human1[lab_name]).sum()}")

HMGU column filled with boolean values
True matches: 122
False (no match): 8334
CEMBIO column filled with boolean values
True matches: 283
False (no match): 8173
AFEKTA column filled with boolean values
True matches: 140
False (no match): 8316
ICL column filled with boolean values
True matches: 116
False (no match): 8340
AUTh column filled with boolean values
True matches: 150
False (no match): 8306


# 1.1- Data on the HUMAN1 graph
After removing compartments and removing isolated nodes
we get the compund graph from the HUMAN1 GEM

In [11]:
import networkx as nx
from matplotlib import pyplot as plt
from upsetplot import UpSet, from_memberships

graph = nx.read_gml(graph_gml)
n = graph.number_of_nodes()
graph_nodes = list(graph.nodes())
node_to_index = {node: idx for idx, node in enumerate(graph_nodes)}

print(f"Graph loaded with {n} nodes.")
print(f"Graph has {graph.number_of_edges()} edges.")
print(f"Graph is connected: {nx.is_connected(graph)}")

Graph loaded with 3240 nodes.
Graph has 7254 edges.
Graph is connected: True


### Then matching the devices metabolites to the graph nodes 

In [12]:
device_indices = {}

for device_name in ['Mitra', 'Capitainer', 'Blood', 'Plasma']:
    device_filtered = df_human1[df_human1[device_name] == True]
    device_ids = set(device_filtered.index.tolist())

    selected_indices = [node_to_index[node_id] for node_id in device_ids if node_id in node_to_index]
    device_indices[device_name] = selected_indices
    print(f"{device_name} indices mapped to graph nodes: {len(selected_indices)} found.")

Mitra indices mapped to graph nodes: 77 found.
Capitainer indices mapped to graph nodes: 75 found.
Blood indices mapped to graph nodes: 70 found.
Plasma indices mapped to graph nodes: 73 found.


In [ ]:
# Build membership list for each observed graph node
all_nodes = sorted(set().union(*device_indices.values()))
memberships = [
    tuple(sorted([device for device, indices in device_indices.items() if node_idx in indices]))
    for node_idx in all_nodes
]

upset_data = from_memberships(memberships)

plt.figure(figsize=(10, 6))
upset = UpSet(upset_data, subset_size='count', show_counts=True, sort_by='degree')
upset.plot()
plt.title('Device intersections for observed metabolites')
plt.tight_layout()
plt.savefig(data_folder / "figs" / "device_intersections_upset.png")
plt.show()

### Matching different labs metabolites on the graph nodes

In [13]:
lab_indices = {}
for lab_name in ['HMGU', 'CEMBIO', 'AFEKTA', 'ICL', 'AUTh']:
    lab_filtered = df_human1[df_human1[lab_name] == True]
    lab_ids = set(lab_filtered.index.tolist())

    selected_indices = [node_to_index[node_id] for node_id in lab_ids if node_id in node_to_index]
    lab_indices[lab_name] = selected_indices
    print(f"{lab_name} indices mapped to graph nodes: {len(selected_indices)} found.")

HMGU indices mapped to graph nodes: 25 found.
CEMBIO indices mapped to graph nodes: 57 found.
AFEKTA indices mapped to graph nodes: 24 found.
ICL indices mapped to graph nodes: 27 found.
AUTh indices mapped to graph nodes: 21 found.


In [ ]:
# Build membership list for each observed graph node
all_lab_nodes = sorted(set().union(*lab_indices.values()))
lab_memberships = [
    tuple(sorted([lab for lab, indices in lab_indices.items() if node_idx in indices]))
    for node_idx in all_lab_nodes]
lab_upset_data = from_memberships(lab_memberships)
plt.figure(figsize=(10, 6))
lab_upset = UpSet(lab_upset_data, subset_size='count', show_counts=True, sort_by='degree')
lab_upset.plot()
plt.title('Lab intersections for observed metabolites')
plt.tight_layout()
plt.savefig(data_folder / "figs" / "lab_intersections_upset.png")
plt.show()

### Degree distribution for devices

In [ ]:
degrees = [d for _, d in graph.degree()]
unique, counts = np.unique(degrees, return_counts=True)

degree_devices = {
    device_name: np.array(degrees)[device_indices[device_name]] for device_name in devices.keys()
    }
freq = counts / counts.sum()

plt.figure(figsize=(6,4))
plt.loglog(unique, freq, 'o', markersize=4, label='all degrees', alpha=0.8)
for device_name, device_degrees in degree_devices.items():
    unique_dev, counts_dev = np.unique(device_degrees, return_counts=True)
    freq_dev = counts_dev / counts_dev.sum()
    plt.loglog(unique_dev, freq_dev, 'o', markersize=6, label=device_name, alpha=0.8)
plt.xlabel('Degree (k)')
plt.ylabel('Frequency P(k)')
plt.title('Degree Distribution of Graph and Devices')
plt.legend()
plt.grid(True, which="both", ls="--", linewidth=0.5)

plt.savefig(data_folder / 'figs' / "degree_distribution.png", dpi=300, bbox_inches='tight')
plt.show()

### Degree distribution for different labs

In [ ]:
degree_labs = {
    lab_name: np.array(degrees)[lab_indices[lab_name]] for lab_name in labs.keys()
    }
plt.figure(figsize=(6,4))
plt.loglog(unique, freq, 'o', markersize=4, label='all degrees', alpha=0.8)
for lab_name, lab_degrees in degree_labs.items():
    unique_lab, counts_lab = np.unique(lab_degrees, return_counts=True)
    freq_lab = counts_lab / counts_lab.sum()
    plt.loglog(unique_lab, freq_lab, 'o', markersize=6, label=lab_name, alpha=0.8)
plt.xlabel('Degree (k)')
plt.ylabel('Frequency P(k)')
plt.title('Degree Distribution of Graph and Labs')
plt.legend()
plt.grid(True, which="both", ls="--", linewidth=0.5)
plt.savefig(data_folder / 'figs' / "degree_distribution_labs.png", dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
observed_nodes = [
    n for n in df_human1.index
    if df_human1.loc[n, "Consortium"] and n in graph
    ]
observed_indices = [node_to_index[node_id] for node_id in observed_nodes]

connector_indices = set(observed_indices)

for i, source_node in enumerate(observed_nodes):
    for target_node in observed_nodes[i + 1 :]:
        try:
            path = nx.shortest_path(graph, source=source_node, target=target_node)
            connector_indices.update(node_to_index[node_id] for node_id in path)
        except nx.NetworkXNoPath:
            pass

print(f"Total observed nodes (including connectors): {len(connector_indices)}")
connector_nodes = [graph_nodes[idx] for idx in connector_indices]

G_observed = graph.subgraph(connector_nodes).copy()
print(G_observed)
print(f"Nodes: {G_observed.number_of_nodes()}")
print(f"Edges: {G_observed.number_of_edges()}")

#### Voronoi graph regions and observations

In [ ]:
from scipy.spatial import Voronoi, voronoi_plot_2d

# 1. Get 2D coordinates for all graph nodes
pos = nx.spring_layout(graph, seed=42, iterations=200)

# 2. Observed nodes = union across devices
observed_nodes = sorted(set().union(*[set(v) for v in device_indices.values()]))
observed_nodes = [n for n in observed_nodes if n in graph]

points = np.array([pos[n] for n in observed_nodes])

# 1. Get 2D coordinates for all graph nodes
pos = nx.spring_layout(graph, seed=42, iterations=200)

# 2. Observed nodes = union across devices
observed_indices = sorted(set().union(*[set(v) for v in device_indices.values()]))
observed_nodes = [graph_nodes[idx] for idx in observed_indices if idx < len(graph_nodes)]
points = np.array([pos[n] for n in observed_nodes])

if points.ndim != 2 or points.shape[0] < 2:
    raise ValueError(f"Need at least 2 observed points for Voronoi, got {points.shape}")

# 3. Compute Voronoi from observed metabolites
vor = Voronoi(points)

# 4. Plot Voronoi regions
fig, ax = plt.subplots(figsize=(10, 10))

voronoi_plot_2d(
    vor,
    ax=ax,
    show_vertices=False,
    show_points=False,
    line_width=0.6,
    line_alpha=0.8
)

# 5. Plot all Human1 nodes in background
all_xy = np.array([pos[n] for n in graph.nodes()])
ax.scatter(
    all_xy[:, 0],
    all_xy[:, 1],
    s=3,
    alpha=0.25,
    color="gray"
)

# 6. Plot observed metabolites
ax.scatter(
    points[:, 0],
    points[:, 1],
    s=18,
    color="black",
    zorder=5
)

ax.set_title("Geometric Voronoi regions around observed metabolites")
ax.set_axis_off()
plt.tight_layout()
plt.savefig(data_folder / 'figs' / 'voronoi_plot_2d.png', dpi=300, bbox_inches='tight')
plt.show()

# 2- Model application
Importing the $H$ function and calculate energy for different k or devices

In [14]:
from etc.hamiltonian import Hamiltonian
from etc.optimization import EnergyOptimizer

#### Define H object and parameters $\mu$ and $\gamma$

In [15]:
H = Hamiltonian(graph)
optimizer = EnergyOptimizer(H)
mu = H.mu_density_aware(graph)
gamma = H.gamma_balancer(mu=mu)

print(f"Gamma value set to: {gamma:.4f}")
print(f"Mu value set to: {mu:.4f}")

Gamma value set to: 2.5562
Mu value set to: 0.9986


In [16]:
distance_matrix = nx.floyd_warshall_numpy(graph)

#### Define devices'and labs' dictionary with values of $\mathcal{H}$

We adjust $\gamma$ here for a fair evaluation the estimation of this new value is done below this section

In [17]:
device_h_values = {}
for device_name in devices.keys():
    h,t1,t2 = H.compute(device_indices[device_name], gamma=gamma, mu=mu)
    device_h_values[device_name] = (h, t1, t2)
    
print("H values computed for each device:")
for device_name, values in device_h_values.items():
    print(f"{device_name}: H={values[0]:.4f}, t1={values[1]:.4f}, t2={values[2]:.4f}, k={len(device_indices[device_name])}")
print(f"gamma = {gamma:.4f}, mu = {mu:.4f}")

H values computed for each device:
Capitainer: H=415.8094, t1=-41.9419, t2=457.7514, k=75
Mitra: H=445.3879, t1=-44.9378, t2=490.3257, k=77
Blood: H=349.3301, t1=-35.9502, t2=385.2803, k=70
Plasma: H=375.3186, t1=-35.9502, t2=411.2688, k=73
gamma = 2.5562, mu = 0.9986


In [18]:
labs_h_values = {}
for lab_name in labs.keys():
    h,t1,t2 = H.compute(lab_indices[lab_name], gamma=gamma, mu=mu)
    labs_h_values[lab_name] = (h, t1, t2)
print("H values computed for each lab:")
for lab_name, values in labs_h_values.items():
    print(f"{lab_name}: H={values[0]:.4f}, t1={values[1]:.4f}, t2={values[2]:.4f}, k={len(lab_indices[lab_name])}")
print(f"gamma = {gamma:.4f}, mu = {mu:.4f}")

H values computed for each lab:
HMGU: H=50.2992, t1=-5.9917, t2=56.2909, k=25
CEMBIO: H=239.3699, t1=-25.9641, t2=265.3339, k=57
AFEKTA: H=43.4951, t1=-3.9945, t2=47.4896, k=24
ICL: H=39.8002, t1=-3.9945, t2=43.7947, k=27
AUTh: H=37.4367, t1=-2.9959, t2=40.4325, k=21
gamma = 2.5562, mu = 0.9986


#### Sampling on k random selections to see $\mathcal{H}$ variation

In [19]:
energies_devices = {}
for device_name in devices.keys():
    energies = optimizer.sampling_energy(
        n=n,k=len(device_indices[device_name]), 
        gamma=gamma, mu=mu,
        n_samples=1000,n_workers=8,seed=42)[0]
    print(f"{device_name} Min energy: {energies.min()}, Max energy: {energies.max()}")
    energies_devices[device_name] = energies

Capitainer Min energy: 160.58549049812268, Max energy: 320.7889734139434
Mitra Min energy: 166.11648092185803, Max energy: 320.6849222454466
Blood Min energy: 129.89571481189336, Max energy: 303.99768085834194
Plasma Min energy: 140.69080622224448, Max energy: 302.4808433025571


In [20]:
energies_labs = {}
for lab_name in labs.keys():
    energies = optimizer.sampling_energy(
        n=n,k=len(lab_indices[lab_name]), 
        gamma=gamma, mu=0.9,
        n_samples=1000,n_workers=8,seed=42)[0]
    print(f"{lab_name} Min energy: {energies.min()}, Max energy: {energies.max()}")
    energies_labs[lab_name] = energies

HMGU Min energy: 12.10277351154109, Max energy: 49.13688994008019
CEMBIO Min energy: 74.06642656095322, Max energy: 188.77463629132936
AFEKTA Min energy: 12.955634513206713, Max energy: 36.25307015561668
ICL Min energy: 16.16170691224078, Max energy: 48.50006239745666
AUTh Min energy: 8.681607835154786, Max energy: 36.0181915754168


In [ ]:
def balanced_nodes_sample(
    optimizer,
    graph,
    k: int,
    distance_matrix: np.ndarray,
    seed: int = 42,
    anchor_fraction: float = 0.25,
):
    distance_matrix = np.asarray(distance_matrix, dtype=float)
    n = len(graph.nodes)

    if not 0 < anchor_fraction <= 1:
        raise ValueError("anchor_fraction must be between 0 and 1.")

    rng = np.random.default_rng(seed)

    # For anchor_fraction = 0.25, this gives approximately k/4 anchors.
    n_anchors = max(
        1,
        min(k, int(round(k * anchor_fraction))),
    )

    # Use your existing farthest-node method unchanged.
    anchor_nodes = optimizer.farthest_nodes_sample(
        graph=graph,
        k=n_anchors,
        seed=seed,
        distance_matrix=distance_matrix,
    )

    anchor_nodes = np.asarray(anchor_nodes, dtype=int)

    selected = set(anchor_nodes.tolist())

    clusters = {
        int(anchor): [int(anchor)]
        for anchor in anchor_nodes
    }
    # For each anchor, sort all nodes from closest to farthest.
    candidate_orders = {
        int(anchor): np.argsort(
            distance_matrix[int(anchor)],
            kind="stable",
        )
        for anchor in anchor_nodes
    }

    # Current position inside each anchor's candidate list.
    candidate_pointers = {
        int(anchor): 0
        for anchor in anchor_nodes
    }

    # Randomize which anchor receives an extra node first when the
    # number of remaining nodes is not divisible by n_anchors.
    anchor_cycle = anchor_nodes.copy()
    rng.shuffle(anchor_cycle)
    while len(selected) < k:
        nodes_added_this_round = 0

        for anchor in anchor_cycle:
            if len(selected) >= k:
                break

            anchor = int(anchor)
            candidate_order = candidate_orders[anchor]
            pointer = candidate_pointers[anchor]

            # Skip the anchor itself and any node already selected
            # by another cluster.
            while (
                pointer < n
                and int(candidate_order[pointer]) in selected
            ):
                pointer += 1

            candidate_pointers[anchor] = pointer

            if pointer >= n:
                continue

            candidate = int(candidate_order[pointer])

            selected.add(candidate)
            clusters[anchor].append(candidate)

            candidate_pointers[anchor] += 1
            nodes_added_this_round += 1

        if nodes_added_this_round == 0:
            raise RuntimeError(
                "No additional candidates could be selected."
            )

    selected_nodes = np.asarray(sorted(selected), dtype=int)

    return selected_nodes, anchor_nodes, clusters

In [ ]:
balanced_sample, anchor_nodes, clusters = balanced_nodes_sample(
    optimizer=optimizer,
    graph=graph,
    k=75,
    distance_matrix=distance_matrix,
    seed=42,
    anchor_fraction=0.25,
)

print("Total selected nodes:", len(balanced_sample))
print("Anchor nodes:", anchor_nodes)
print(balanced_sample)


In [ ]:
S0_balanced = np.zeros(len(graph.nodes), dtype=int)
S0_balanced[balanced_sample] = 1

assert S0_balanced.sum() == 75
assert len(S0_balanced) == len(graph.nodes)

In [ ]:
Emin = optimizer.min_energy_annealing(S0_balanced, mu=mu, gamma=gamma,
                            Tmax=1.0, Tmin=1e-6,
                            seed=42, steps=100, n_workers=8)

In [ ]:
H.compute(balanced_sample, gamma=gamma, mu=mu)

In [25]:
S1HMGU = optimizer.close_nodes_sample(graph=graph, k=25)

In [26]:
H.compute(S1HMGU, gamma=gamma, mu=mu)

(213.75631367706083, -23.96682103529896, 237.72313471235978)

In [ ]:
# Get position indices of selected nodes in the graph (0-based positions, integers)
position_indices = optimizer.close_nodes_sample(graph=graph, k=25)
print(f"Position indices: {position_indices}")
print(f"These are 0-based position integers: {type(position_indices[0])}")

#### Finding $E_{max}$
Since the graph seems to be more sparse that dense, and the energy sampling is mostly in the positives values
for maximization we will look for the farthest nodes in the graph as an initial condition to maxime

In [ ]:
S0Capitainer = optimizer.create_S0_mask(
    optimizer.farthest_nodes_sample(graph=graph, k=75, seed=42,distance_matrix=distance_matrix),
    graph.nodes())
S0Mitra = optimizer.create_S0_mask(
    optimizer.farthest_nodes_sample(graph=graph, k=77, seed=42,distance_matrix=distance_matrix),
    graph.nodes())
S0Blood = optimizer.create_S0_mask(
    optimizer.farthest_nodes_sample(graph=graph, k=70, seed=42,distance_matrix=distance_matrix),
    graph.nodes())
S0Plasma = optimizer.create_S0_mask(
    optimizer.farthest_nodes_sample(graph=graph, k=73, seed=42,distance_matrix=distance_matrix),
    graph.nodes())

In [ ]:
S0HMGU = optimizer.create_S0_mask(
    optimizer.farthest_nodes_sample(graph=graph, k=25, seed=42,distance_matrix=distance_matrix),
    graph.nodes())
S0CEMBIO = optimizer.create_S0_mask(
    optimizer.farthest_nodes_sample(graph=graph, k=57, seed=42,distance_matrix=distance_matrix),
    graph.nodes())
S0AFEKTA = optimizer.create_S0_mask(
    optimizer.farthest_nodes_sample(graph=graph, k=24, seed=42,distance_matrix=distance_matrix),
    graph.nodes())
S0ICL = optimizer.create_S0_mask(
    optimizer.farthest_nodes_sample(graph=graph, k=27, seed=42,distance_matrix=distance_matrix),
    graph.nodes())
S0AUTh = optimizer.create_S0_mask(
    optimizer.farthest_nodes_sample(graph=graph, k=21, seed=42,distance_matrix=distance_matrix),
    graph.nodes())

In [ ]:
# Dictionary of configurations: (S0_config, gamma, name)
configs = {
    'cap': (S0Capitainer, 0.02),
    'mitra': (S0Mitra, 0.02),
    'blood': (S0Blood, 0.02),
    'plasma': (S0Plasma, 0.02),
    'hmgu': (S0HMGU, 0.01),
    'afekta': (S0AFEKTA, 0.01),
    'cembio': (S0CEMBIO, 0.01),
    'icl': (S0ICL, 0.01),
    'auth': (S0AUTh, 0.01),
}

# Compute max energy for all configurations
emax_values = {}
for name, (S0_config, gamma) in configs.items():
    emax_values[f'Emax_{name}'] = optimizer.max_energy_annealing(
        S0_config, gamma=gamma, mu=mu, steps=10, n_workers=8, seed=42
    )[1]

# Extract individual variables for backward compatibility
Emax_cap = emax_values['Emax_cap']
Emax_mitra = emax_values['Emax_mitra']
Emax_blood = emax_values['Emax_blood']
Emax_plasma = emax_values['Emax_plasma']
Emax_hmgu = emax_values['Emax_hmgu']
Emax_afekta = emax_values['Emax_afekta']
Emax_cembio = emax_values['Emax_cembio']
Emax_icl = emax_values['Emax_icl']
Emax_auth = emax_values['Emax_auth']

print("All Emax values computed successfully")

In [ ]:
H.compute(optimizer.farthest_nodes_sample(graph=graph, k=25, seed=42,distance_matrix=distance_matrix), gamma=gamma, mu=mu)

In [ ]:
# Import and use the refactored EnergyOptimizer class
from etc.optimization import EnergyOptimizer

# Initialize with your Hamiltonian
optimizer = EnergyOptimizer(H)

# Example: Sample energy values
energies, min_sample, max_sample = optimizer.sampling_energy(
    n=100, k=10, gamma=0.5, mu=0.1, 
    n_samples=10000, seed=42, n_workers=8
)
print(f"Min energy: {energies.min()}, Max energy: {energies.max()}")

# Example: Minimize energy using parallel annealing
S0_config = optimizer.create_S0_mask([0, 1, 2, 3, 4, 5, 6, 7, 8, 9], range(100))
S_min, E_min, history = optimizer.min_energy_annealing(
    S0_config, mu=0.1, gamma=0.5, steps=10000, n_workers=8
)
print(f"Minimum energy found: {E_min}")

# Example: Maximize energy using parallel annealing
S_max, E_max, history = optimizer.max_energy_annealing(
    S0_config, mu=0.1, gamma=0.5, steps=10000, n_workers=8
)
print(f"Maximum energy found: {E_max}")

#### Define H object and parameters $\mu$ and $\gamma$
- method to binarize index lists

In [ ]:
H = Hamiltonian(graph)
coverage = Coverage(H)

mu = H.mu_density_aware(graph)
gamma = H.gamma_balancer(mu=mu)

print(f"Gamma value set to: {gamma:.4f}")
print(f"Mu value set to: {mu:.4f}")

def S0(idx,nodes):
    mask = np.zeros(len(nodes), dtype=bool)
    mask[idx] = True
    return mask

#### Define devices' dictionary with values of $\mathcal{H}$

In [ ]:
device_h_values = {}
for device_name in devices.keys():
    h,t1,t2 = H.compute(device_indices[device_name], gamma=0.23, mu=mu)
    device_h_values[device_name] = (h, t1, t2)
    
print("H values computed for each device:")
for device_name, values in device_h_values.items():
    print(f"{device_name}: H={values[0]:.4f}, t1={values[1]:.4f}, t2={values[2]:.4f}, k={len(device_indices[device_name])}")

In [ ]:
labs_h_values = {}
for lab_name in labs.keys():
    h,t1,t2 = H.compute(lab_indices[lab_name], gamma=0.18, mu=0.9)
    labs_h_values[lab_name] = (h, t1, t2)
print("H values computed for each lab:")
for lab_name, values in labs_h_values.items():
    print(f"{lab_name}: H={values[0]:.4f}, t1={values[1]:.4f}, t2={values[2]:.4f}, k={len(lab_indices[lab_name])}")

In [ ]:
from etc.coverage import Coverage
from joblib import Parallel, delayed
from tqdm.auto import tqdm
coverage = Coverage(H)
from random import choices

In [ ]:
def sampling_energy(n, k, gamma, mu, n_samples=10000, seed=42, n_workers=8):

    import concurrent.futures
    import os

  
    n_workers = max(1, min(n_workers, n_samples))

    seed_seq = np.random.SeedSequence(seed)
    child_seeds = seed_seq.spawn(n_workers)

    chunk_sizes = [
        n_samples // n_workers + (1 if i < n_samples % n_workers else 0)
        for i in range(n_workers)
    ]

    def worker(chunk_size, child_seed):
        rng_worker = np.random.default_rng(child_seed)
        energies_chunk = np.empty(chunk_size, dtype=float)
        samples_chunk = np.empty((chunk_size, k), dtype=int)
        for idx in range(chunk_size):
            sample = rng_worker.choice(n, size=k, replace=False)
            E = H.compute(sample, gamma=gamma, mu=mu)[0]
            energies_chunk[idx] = abs(E)
            samples_chunk[idx] = sample
        return energies_chunk, samples_chunk

    tasks = [
        (chunk_sizes[i], child_seeds[i])
        for i in range(n_workers)
        if chunk_sizes[i] > 0
    ]

    if len(tasks) == 1:
        results = [worker(*tasks[0])]
    else:
        with concurrent.futures.ThreadPoolExecutor(max_workers=len(tasks)) as executor:
            futures = [executor.submit(worker, chunk_size, child_seed) for chunk_size, child_seed in tasks]
            results = [future.result() for future in futures]

    energies = np.concatenate([r[0] for r in results])
    samples = np.concatenate([r[1] for r in results], axis=0)

    return energies, samples[energies.argmin()], samples[energies.argmax()]

### Find $\gamma$ to balance $\mathcal{H}$ for devices

In [ ]:
def gamma_annealing_sweep(device_name, gamma_grid, steps=150000, seed=42, Tmax=1.0, Tmin=1e-50, cooloing=0.995,device=True):
    if device:
        device_mask = S0(device_indices[device_name], node_to_index.values())
        rows = []
    else:
        device_mask = S0(lab_indices[device_name], node_to_index.values())
        rows = []

    for gamma_candidate in gamma_grid:
        best_mask, best_energy, history = coverage.min_energy_anneling(
            S0=device_mask,
            mu=mu,
            gamma=float(gamma_candidate),
            Tmax=Tmax,
            Tmin=Tmin,
            cooloing=cooloing,
            seed=seed,
            steps=steps,
        )
        best_indices = np.where(best_mask == 1)[0]
        best_h, _, _ = H.compute(best_indices, mu=mu, gamma=float(gamma_candidate))
        rows.append(
            {
                "device": device_name,
                "k": int(device_mask.sum()),
                "gamma": float(gamma_candidate),
                "best_abs_H": float(best_energy),
                "best_signed_H": float(best_h),
                "history_min": float(np.min(history)) if len(history) > 0 else float('nan'),
                "history_last": float(history[-1]) if len(history) > 0 else float('nan'),
            }
        )

    return pd.DataFrame(rows)

In [ ]:
# Estimate a per-device gamma from the current H balance without creating the later energy arrays yet.
device_gamma_estimates = {}
for name, (h_val, t1, t2) in device_h_values.items():
    if t2 is None or abs(t2) < 1e-12:
        est = float(gamma)
    else:
        est = float((-t1 * float(gamma)) / float(t2))
    device_gamma_estimates[name] = max(1e-3, est)

scan_frames = []
for name in device_indices.keys():
    gamma_estimate = device_gamma_estimates[name]
    gamma_grid = np.linspace(max(0.01, 0.1 * gamma_estimate), max(0.01, 3.0 * gamma_estimate), 18)
    scan_frames.append(gamma_annealing_sweep(name, gamma_grid, steps=1500000, seed=42,Tmin=1e-1000))

annealing_scan = pd.concat(scan_frames, ignore_index=True)

best_gamma_by_device = (
    annealing_scan.sort_values(["device", "best_abs_H"])
    .groupby("device", as_index=False)
    .first()[["device", "k", "gamma", "best_abs_H", "best_signed_H"]]
)

print(pd.DataFrame({"device": list(device_gamma_estimates.keys()),
                    "gamma_estimate": list(device_gamma_estimates.values())}).to_string(index=False))
print(best_gamma_by_device.to_string(index=False))

fig, ax = plt.subplots(figsize=(10, 6))
for device_name, device_df in annealing_scan.groupby("device"):
    ax.plot(device_df["gamma"], device_df["best_signed_H"], marker="o", linewidth=1.5, label=device_name)

ax.axhline(0.0, color="black", linewidth=1.0, linestyle="--")
ax.set_xlabel('s')
ax.set_ylabel('min E')
ax.set_title(r'Gamma sweep for device-specific k $(\gamma = s \cdot \gamma_{estimate})$')
ax.legend()
ax.grid(True, alpha=0.25)
out_dir = data_folder / 'figs'
plt.savefig(out_dir / 'gamma_sweep_plot.png', dpi=400, bbox_inches='tight')
plt.show()

### Find $\gamma$ to balance $\mathcal{H}$ for labs observations

In [ ]:
# Estimate a per-device gamma from the current H balance without creating the later energy arrays yet.
lab_gamma_estimates = {}
for name, (h_val, t1, t2) in labs_h_values.items():
    if t2 is None or abs(t2) < 1e-12:
        est = float(gamma)
    else:
        est = float((-t1 * float(gamma)) / float(t2))
    lab_gamma_estimates[name] = max(1e-3, est)

scan_frames = []
for name in lab_indices.keys():
    gamma_estimate = lab_gamma_estimates[name]
    gamma_grid = np.linspace(max(0.01, 0.1 * gamma_estimate), max(0.01, 3.0 * gamma_estimate), 18)
    scan_frames.append(gamma_annealing_sweep(name, gamma_grid, steps=1500000, seed=42,Tmin=1e-1000,device=False))

In [ ]:
annealing_scan_labs = pd.concat(scan_frames, ignore_index=True)

best_gamma_by_lab = (
    annealing_scan_labs.sort_values(["device", "best_abs_H"])
    .groupby("device", as_index=False)
    .first()[["device", "k", "gamma", "best_abs_H", "best_signed_H"]]
)
print(pd.DataFrame({"lab": list(lab_gamma_estimates.keys()),
                    "gamma_estimate": list(lab_gamma_estimates.values())}).to_string(index=False))
print(best_gamma_by_lab.to_string(index=False))

fig, ax = plt.subplots(figsize=(10, 6))
for device_name, device_df in annealing_scan_labs.groupby("device"):
    ax.plot(device_df["gamma"], device_df["best_signed_H"], marker="o", linewidth=1.5, label=device_name)
ax.axhline(0.0, color="black", linewidth=1.0, linestyle="--")
ax.set_xlabel('s')
ax.set_ylabel('min E')
ax.set_title(r'Gamma sweep for device-specific k $(\gamma = s \cdot \gamma_{estimate})$')
ax.legend()
ax.grid(True, alpha=0.25)
plt.savefig(out_dir / 'gamma_sweep_plot_labs.png', dpi=400, bbox_inches='tight')
plt.show()

#### Binary vectors $s_{i}$ for devices

In [ ]:
gamma = 0.23
S0_Capitainer = S0(device_indices['Capitainer'], graph_nodes)
S0_Mitra = S0(device_indices['Mitra'], graph_nodes)
S0_Blood = S0(device_indices['Blood'], graph_nodes)
S0_Plasma = S0(device_indices['Plasma'], graph_nodes)

In [ ]:
gamma_labs = 0.2
S0_HMGU = S0(lab_indices['HMGU'], graph_nodes)
S0_CEMBIO = S0(lab_indices['CEMBIO'], graph_nodes)
S0_AFEKTA = S0(lab_indices['AFEKTA'], graph_nodes)
S0_ICL = S0(lab_indices['ICL'], graph_nodes)

# 2.1 - Energy evaluation and random sampling
Random sampling for different k

In [ ]:
def sampling_energy(n, k, gamma, mu, n_samples=10000, seed=42, n_workers=None):

    import concurrent.futures
    import os

    if n_workers is None:
        n_workers = max(1, min(n_samples, os.cpu_count() or 1))
    else:
        n_workers = max(1, min(n_workers, n_samples))

    seed_seq = np.random.SeedSequence(seed)
    child_seeds = seed_seq.spawn(n_workers)

    chunk_sizes = [
        n_samples // n_workers + (1 if i < n_samples % n_workers else 0)
        for i in range(n_workers)
    ]

    def worker(chunk_size, child_seed):
        rng_worker = np.random.default_rng(child_seed)
        energies_chunk = np.empty(chunk_size, dtype=float)
        samples_chunk = np.empty((chunk_size, k), dtype=int)
        for idx in range(chunk_size):
            sample = rng_worker.choice(n, size=k, replace=False)
            E = H.compute(sample, gamma=gamma, mu=mu)[0]
            energies_chunk[idx] = abs(E)
            samples_chunk[idx] = sample
        return energies_chunk, samples_chunk

    tasks = [
        (chunk_sizes[i], child_seeds[i])
        for i in range(n_workers)
        if chunk_sizes[i] > 0
    ]

    if len(tasks) == 1:
        results = [worker(*tasks[0])]
    else:
        with concurrent.futures.ThreadPoolExecutor(max_workers=len(tasks)) as executor:
            futures = [executor.submit(worker, chunk_size, child_seed) for chunk_size, child_seed in tasks]
            results = [future.result() for future in futures]

    energies = np.concatenate([r[0] for r in results])
    samples = np.concatenate([r[1] for r in results], axis=0)

    return energies, samples[energies.argmin()], samples[energies.argmax()]


In [ ]:
# Example: parallel random sampling with 8 workers
energies, best_sample, worst_sample = sampling_energy(
    n=H.n,
    k=25,
    gamma=0.2,
    mu=0.9,
    n_samples=10000,
    seed=42,
    n_workers=8,
)
print('best energy sample size:', best_sample)
print('worst energy sample size:', worst_sample)

The random sampling takes around 2h

In [ ]:
sampling_results = {}
for device_name in devices.keys():
     k = len(device_indices[device_name])
     energies, best_sample, worst_sample = sampling_energy(n, k, gamma=0.23, mu=mu, n_samples=100000, seed=42, n_workers=8)
     sampling_results[device_name] = (energies, best_sample, worst_sample)
     print(
         f"{device_name}: Best energy={energies.min():.4f}, Worst energy={energies.max():.4f}")

In [ ]:
sampling_results_labs = {}
for lab_name in labs.keys():
     k = len(lab_indices[lab_name])
     energies, best_sample, worst_sample = sampling_energy(n, k, gamma=0.2, mu=0.9, n_samples=100000, seed=42, n_workers=8)
     sampling_results_labs[lab_name] = (energies, best_sample, worst_sample)
     print(
         f"{lab_name}: Best energy={energies.min():.4f}, Worst energy={energies.max():.4f}")

Store it in a txt file to fast experiments

In [ ]:
# # Store the sampling results in a txt file
results_file = data_folder /"generated" / "sampling_results.txt"
# with open(results_file, 'w') as f:
#     for device_name, (energies, best_sample, worst_sample) in sampling_results.items():
#         f.write(f"{device_name}:\n")
#         f.write(f"Best energy: {energies.min():.4f}\n")
#         f.write(f"Worst energy: {energies.max():.4f}\n")
#         f.write(f"Best sample indices: {best_sample}\n")
#         f.write(f"Worst sample indices: {worst_sample}\n\n")

In [ ]:
with open(results_file, 'r') as f:
    contents = f.read()

best_sample_indices_by_device = {}
worst_sample_indices_by_device = {}
best_sample_indices_raw_by_device = {}
worst_sample_indices_raw_by_device = {}

for section in contents.strip().split("\n\n"):
    section = section.strip()
    if not section:
        continue

    header_match = re.match(r"^(.*?):", section)
    if header_match is None:
        continue

    device_name = header_match.group(1).strip()
    best_match = re.search(r"Best sample indices:\s*(\[[\s\S]*?\])", section)
    worst_match = re.search(r"Worst sample indices:\s*(\[[\s\S]*?\])", section)

    best_raw = best_match.group(1) if best_match else None
    worst_raw = worst_match.group(1) if worst_match else None

    best_sample_indices_raw_by_device[device_name] = best_raw
    worst_sample_indices_raw_by_device[device_name] = worst_raw
    best_sample_indices_by_device[device_name] = [int(value) for value in re.findall(r"-?\d+", best_raw)] if best_raw else None
    worst_sample_indices_by_device[device_name] = [int(value) for value in re.findall(r"-?\d+", worst_raw)] if worst_raw else None


best_sample_indices_capitainer = best_sample_indices_by_device["Capitainer"]
best_sample_indices_mitra = best_sample_indices_by_device["Mitra"]
best_sample_indices_blood = best_sample_indices_by_device["Blood"]
best_sample_indices_plasma = best_sample_indices_by_device["Plasma"]

worst_sample_indices_capitainer = worst_sample_indices_by_device["Capitainer"]
worst_sample_indices_mitra = worst_sample_indices_by_device["Mitra"]
worst_sample_indices_blood = worst_sample_indices_by_device["Blood"]
worst_sample_indices_plasma = worst_sample_indices_by_device["Plasma"]

### Min Energy for devices
- Capitainer
- Mitra
- Blood
- Plasma

In [ ]:
S0_minCapitainer = S0(S0_Capitainer, graph_nodes)
min_C = coverage.min_energy_anneling(S0_minCapitainer, mu=mu, gamma=gamma,steps=100000000,Tmin=1e-1000)
print(f"Min energy for Capitainer: {min_C[1]:.4f}")
S0_minMitra = S0(sampling_results["Mitra"][1], graph_nodes)
min_M = coverage.min_energy_anneling(S0_minMitra, mu=mu, gamma=gamma,steps=100000000,Tmin=1e-500)
print(f"Min energy for Mitra: {min_M[1]:.4f}")

S0_minBlood = S0(sampling_results["Blood"][1], graph_nodes)
min_B = coverage.min_energy_anneling(S0_minBlood, mu=mu, gamma=gamma,steps=100000000, seed=40,Tmin=1e-50)
print(f"Min energy for Blood: {min_B[1]:.4f}")

S0_minPlasma = S0(sampling_results["Plasma"][1], graph_nodes)
min_P = coverage.min_energy_anneling(S0_minPlasma, mu=mu, gamma=gamma,steps=100000000, seed=40,Tmin=1e-50)
print(f"Min energy for Plasma: {min_P[1]:.4f}")

In [ ]:
S0_minHMGU = S0(sampling_results_labs["HMGU"][1], graph_nodes)
min_HMGU = coverage.min_energy_anneling(S0_minHMGU, mu=mu, gamma=gamma_labs,steps=100000000, seed=40,Tmin=1e-50)
print(f"Min energy for HMGU: {min_HMGU[1]:.4f}")

S0_minCEMBIO = S0(sampling_results_labs["CEMBIO"][1], graph_nodes)
min_CEMBIO = coverage.min_energy_anneling(S0_minCEMBIO, mu=mu, gamma=gamma_labs,steps=100000000, seed=40,Tmin=1e-50)
print(f"Min energy for CEMBIO: {min_CEMBIO[1]:.4f}")

S0_minAFEKTA = S0(sampling_results_labs["AFEKTA"][1], graph_nodes)
min_AFEKTA = coverage.min_energy_anneling(S0_minAFEKTA, mu=mu, gamma=gamma_labs,steps=100000000, seed=40,Tmin=1e-50)
print(f"Min energy for AFEKTA: {min_AFEKTA[1]:.4f}")

S0_minICL = S0(sampling_results_labs["ICL"][1], graph_nodes)
min_ICL = coverage.min_energy_anneling(S0_minICL, mu=mu, gamma=gamma_labs,steps=100000000, seed=40,Tmin=1e-50)
print(f"Min energy for ICL: {min_ICL[1]:.4f}")

S0_minAUTh = S0(sampling_results_labs["AUTh"][1], graph_nodes)
min_AUTh = coverage.min_energy_anneling(S0_minAUTh, mu=mu, gamma=gamma_labs,steps=100000000, seed=40,Tmin=1e-50)
print(f"Min energy for AUTh: {min_AUTh[1]:.4f}")

In [ ]:
def max_energy_anneling(coverage, S0,
                        mu: float = 1.0, 
                        gamma: float = 1.0,
                        Tmax: float =1.0,
                        Tmin: float =1e-6,
                        cooloing : float = 0.995,
                        seed: int = 42,
                        steps: int = 10000
                        ):
        """
        Maximize E using simulated anneling.
        S0 is the initial subset of nodes closest to the maximum energy configuration.
        S0[i] = 1 if node i is in the subset, 0 otherwise.
        
        Constraints:
        sum(S0) = k is preserved during the optimization.
        
        Returns:
        --------
        S_max : np.ndarray
            The subset of nodes with the maximum energy.
        E_max : float
            The maximum energy value.
        """
        rng = np.random.default_rng(seed)
        
        # initial configuration
        S_current = S0.copy()

        # compute initial energy
        E_current = coverage.energy(S_current, mu=mu, gamma=gamma)

        best_S = S_current.copy()
        best_E = E_current

        # Temperature
        T = Tmax
        history = []

        # Anneling loop
        for _ in range(steps):
            
            proposal_S = S_current.copy()
            #Find occupied and unoccupied indices
            occupied_indices = np.where(proposal_S == 1)[0]
            unoccupied_indices = np.where(proposal_S == 0)[0]

            # Swap move: randomly select one occupied and one unoccupied
            # to swap their states

            remove_node = rng.choice(occupied_indices)
            add_node = rng.choice(unoccupied_indices)

            proposal_S[remove_node] = 0
            proposal_S[add_node] = 1

            # Compute new energy
            proposal_E = coverage.energy(
                proposal_S,
                mu=mu, 
                gamma=gamma
                )
            delta_E = proposal_E - E_current

            # Metropolis criterion
            if delta_E > 0:
                accept = True
            else:
                prob = np.exp(delta_E / T)
                accept = rng.random() < prob
            
            # Accept move
            if accept:
                S_current = proposal_S
                E_current = proposal_E

                # Update best solution
                if E_current > best_E:
                    best_S = S_current.copy()
                    best_E = E_current
        
            # Store history
            history.append((E_current))

            # Cool down
            T *= cooloing
            if T < Tmin:
                break

        return best_S, best_E, history

In [ ]:
S0_maxCapitainer = S0(device_indices['Capitainer'], graph_nodes)
max_C = max_energy_anneling(coverage, S0_maxCapitainer, mu=mu, gamma=gamma, steps=100000000,Tmin=1e-300)
print(f"Max energy for Capitainer: {max_C[1]:.4f}")

S0_maxMitra = S0(device_indices['Mitra'], graph_nodes)
max_M = max_energy_anneling(coverage, S0_maxMitra, mu=mu, gamma=gamma, steps=100000000,Tmin=1e-1000)
print(f"Max energy for Mitra: {max_M[1]:.4f}")

S0_maxBlood = S0(device_indices['Blood'], graph_nodes)
max_B = max_energy_anneling(coverage, S0_maxBlood, mu=mu, gamma=gamma, steps=100000000,Tmin=1e-1000)
print(f"Max energy for Blood: {max_B[1]:.4f}")

S0_maxPlasma = S0(device_indices['Plasma'], graph_nodes)
max_P = max_energy_anneling(coverage, S0_maxPlasma, mu=mu, gamma=gamma, steps=100000000,Tmin=1e-400)
print(f"Max energy for Plasma: {max_P[1]:.4f}")

In [ ]:
S0_maxHMGU = S0(lab_indices['HMGU'], graph_nodes)
max_HMGU = max_energy_anneling(coverage, S0_maxHMGU, mu=mu, gamma=gamma_labs, steps=100000000,Tmin=1e-400)
print(f"Max energy for HMGU: {max_HMGU[1]:.4f}")

S0_maxCEMBIO = S0(lab_indices['CEMBIO'], graph_nodes)
max_CEMBIO = max_energy_anneling(coverage, S0_maxCEMBIO, mu=mu, gamma=gamma_labs, steps=100000000,Tmin=1e-400)
print(f"Max energy for CEMBIO: {max_CEMBIO[1]:.4f}")

S0_maxAFEKTA = S0(lab_indices['AFEKTA'], graph_nodes)
max_AFEKTA = max_energy_anneling(coverage, S0_maxAFEKTA, mu=mu, gamma=gamma_labs, steps=100000000,Tmin=1e-400)
print(f"Max energy for AFEKTA: {max_AFEKTA[1]:.4f}")

S0_maxICL = S0(lab_indices['ICL'], graph_nodes)
max_ICL = max_energy_anneling(coverage, S0_maxICL, mu=mu, gamma=gamma_labs, steps=100000000,Tmin=1e-400)
print(f"Max energy for ICL: {max_ICL[1]:.4f}")

### Average short path distance between nodes

In [ ]:
distance_matrix = H.distance_matrix

def pairwise_distances(device_nodes, dist_matrix):
    distances = []
    for i, j in combinations(device_nodes, 2):
        distances.append(dist_matrix[i, j])
    return np.array(distances)

In [ ]:
distance_matrix

In [ ]:
other = np.concatenate(
[np.asarray(lab_indices[d]) for d in lab_indices.keys() if d != 'CEMBIO']
)

In [ ]:
target_lab = "CEMBIO"

# All values from labs except CEMBIO
other_values = set()

for lab, values in lab_indices.items():
    if lab != target_lab:
        other_values.update(values)

# Values in CEMBIO that are not present in any other lab
unique_to_cembio = [
    value for value in lab_indices[target_lab]
    if value not in other_values
]

print(unique_to_cembio)
print("Number of unique CEMBIO values:", len(unique_to_cembio))

In [ ]:
plt.imshow(distance_matrix[unique_to_cembio, :][:, unique_to_cembio])

In [ ]:
mat_cemb = distance_matrix[unique_to_cembio, :][:, unique_to_cembio]

In [ ]:
pd.Series(mat_cemb.ravel()).value_counts().sort_index()

In [ ]:
degree_values = [d for _, d in graph.degree()]

In [ ]:
degree_values_unique_to_cembio = [degree_values[node] for node in unique_to_cembio]

In [ ]:
degree_values_unique_to_cembio

In [ ]:
degree_values_cembio = [degree_values[node] for node in lab_indices['CEMBIO']]

In [ ]:
degree_values_cembio

In [ ]:
from matplotlib.lines import Line2D


def plot_lab_subgraph_regions(
    g,
    lab_indices,
    graph_nodes,
    *,
    layout="spring",
    seed=42,
    node_size=80,
    observed_size=260,
    alpha_edges=0.25,
):
    """
    Plot a subgraph and highlight observed metabolites by lab.

    Parameters
    ----------
    g : nx.Graph
        Subgraph to visualize.
    lab_indices : dict
        Dictionary {lab_name: list_of_node_indices}.
    graph_nodes : list
        List mapping integer indices to graph node names.
    """

    # Convert lab indices to node names and keep only nodes inside g
    lab_nodes = {
        lab: {
            graph_nodes[idx]
            for idx in indices
            if idx < len(graph_nodes) and graph_nodes[idx] in g
        }
        for lab, indices in lab_indices.items()
    }

    observed_nodes = set().union(*lab_nodes.values()) if lab_nodes else set()

    # Layout
    if layout == "spring":
        pos = nx.spring_layout(g, seed=seed, iterations=300)
    elif layout == "kamada":
        pos = nx.kamada_kawai_layout(g)
    else:
        raise ValueError("layout must be 'spring' or 'kamada'")

    colors = {
        "HMGU": "tab:blue",
        "CEMBIO": "tab:orange",
        "AFEKTA": "tab:green",
        "ICL": "tab:red",
        "AUTh": "tab:purple",
    }

    fig, ax = plt.subplots(figsize=(11, 9))

    # Draw edges
    nx.draw_networkx_edges(
        g,
        pos,
        ax=ax,
        edge_color="gray",
        alpha=alpha_edges,
        width=0.8,
    )

    # Draw all background nodes
    background_nodes = [n for n in g.nodes if n not in observed_nodes]
    nx.draw_networkx_nodes(
        g,
        pos,
        nodelist=background_nodes,
        node_color="lightgray",
        node_size=node_size,
        alpha=0.45,
        linewidths=0,
        ax=ax,
    )

    # Draw observed nodes lab by lab
    for lab, nodes in lab_nodes.items():
        if not nodes:
            continue

        nx.draw_networkx_nodes(
            g,
            pos,
            nodelist=list(nodes),
            node_color=colors.get(lab, "black"),
            node_size=observed_size,
            alpha=0.85,
            edgecolors="black",
            linewidths=0.8,
            label=f"{lab} ({len(nodes)})",
            ax=ax,
        )

    # Highlight nodes shared by more than one lab with larger black ring
    node_lab_count = {}
    for lab, nodes in lab_nodes.items():
        for n in nodes:
            node_lab_count[n] = node_lab_count.get(n, 0) + 1

    shared_nodes = [n for n, c in node_lab_count.items() if c > 1]

    if shared_nodes:
        nx.draw_networkx_nodes(
            g,
            pos,
            nodelist=shared_nodes,
            node_color="none",
            node_size=observed_size + 140,
            edgecolors="black",
            linewidths=2.2,
            ax=ax,
        )

    ax.set_title("Observed metabolites by institution in the selected Human1 subgraph")
    ax.set_axis_off()
    ax.legend(frameon=False, loc="best")

    plt.tight_layout()
    plt.savefig(data_folder / "figs" / "lab_subgraph_regions.png", dpi=300, bbox_inches='tight')
    plt.show()

    return lab_nodes

In [ ]:
observed_indices = set().union(*[set(v) for v in lab_indices.values()])
observed_nodes = {
    graph_nodes[idx]
    for idx in observed_indices
    if idx < len(graph_nodes) and graph_nodes[idx] in graph
}

g_nodes = set(observed_nodes)

for n in observed_nodes:
    g_nodes.update(nx.single_source_shortest_path_length(graph, n, cutoff=1).keys())

g = graph.subgraph(g_nodes).copy()

In [ ]:
lab_nodes_in_g = plot_lab_subgraph_regions(
    g,
    lab_indices,
    graph_nodes,
    layout="spring",
)


In [ ]:
observed = set().union(*lab_nodes_in_g.values())
g = nx.ego_graph(graph, list(observed)[0], radius=2)

In [ ]:
import math
import networkx as nx
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D


def build_observed_neighborhood_subgraph(
    graph,
    lab_indices,
    graph_nodes,
    *,
    radius=1,
):
    """
    Build one shared subgraph containing all observed lab metabolites
    and their local graph neighborhoods.
    """

    lab_nodes = {
        lab: {
            graph_nodes[idx]
            for idx in indices
            if idx < len(graph_nodes) and graph_nodes[idx] in graph
        }
        for lab, indices in lab_indices.items()
    }

    observed_nodes = set().union(*lab_nodes.values())

    g_nodes = set(observed_nodes)

    for node in observed_nodes:
        neighbors = nx.single_source_shortest_path_length(
            graph,
            node,
            cutoff=radius,
        ).keys()
        g_nodes.update(neighbors)

    g = graph.subgraph(g_nodes).copy()

    return g, lab_nodes, observed_nodes


def plot_each_lab_on_same_subgraph(
    graph,
    lab_indices,
    graph_nodes,
    *,
    radius=1,
    layout="spring",
    seed=42,
    ncols=3,
    figsize=(16, 10),
    node_size=18,
    observed_size=100,
):
    """
    Plot one graph per lab using the same subgraph and same node positions.
    """

    g, lab_nodes, observed_nodes = build_observed_neighborhood_subgraph(
        graph,
        lab_indices,
        graph_nodes,
        radius=radius,
    )

    # Same layout for all panels
    if layout == "spring":
        pos = nx.spring_layout(g, seed=seed, iterations=300)
    elif layout == "kamada":
        pos = nx.kamada_kawai_layout(g)
    else:
        raise ValueError("layout must be 'spring' or 'kamada'")

    labs = list(lab_nodes.keys())
    n_labs = len(labs)
    nrows = math.ceil(n_labs / ncols)

    colors = {
        "HMGU": "tab:blue",
        "CEMBIO": "tab:orange",
        "AFEKTA": "tab:green",
        "ICL": "tab:red",
        "AUTh": "tab:purple",
    }

    fig, axes = plt.subplots(
        nrows=nrows,
        ncols=ncols,
        figsize=figsize,
        squeeze=False,
    )

    for ax, lab in zip(axes.flat, labs):
        highlighted_nodes = lab_nodes[lab]

        background_observed = observed_nodes - highlighted_nodes
        background_unobserved = set(g.nodes()) - observed_nodes

        # Edges
        nx.draw_networkx_edges(
            g,
            pos,
            ax=ax,
            edge_color="gray",
            alpha=0.18,
            width=0.6,
        )

        # Unobserved neighborhood nodes
        nx.draw_networkx_nodes(
            g,
            pos,
            nodelist=list(background_unobserved),
            node_color="lightgray",
            node_size=node_size,
            alpha=0.25,
            linewidths=0,
            ax=ax,
        )

        # Observed nodes from other labs
        nx.draw_networkx_nodes(
            g,
            pos,
            nodelist=list(background_observed),
            node_color="gray",
            node_size=node_size * 1.8,
            alpha=0.35,
            linewidths=0,
            ax=ax,
        )

        # Highlight current lab
        nx.draw_networkx_nodes(
            g,
            pos,
            nodelist=list(highlighted_nodes),
            node_color=colors.get(lab, "black"),
            node_size=observed_size,
            alpha=0.9,
            edgecolors="black",
            linewidths=0.6,
            ax=ax,
        )

        ax.set_title(f"{lab} observed metabolites (n={len(highlighted_nodes)})")
        ax.set_axis_off()

    # Hide empty panels
    for ax in axes.flat[n_labs:]:
        ax.set_visible(False)

    legend_elements = [
        Line2D(
            [0], [0],
            marker="o",
            color="w",
            markerfacecolor="lightgray",
            markersize=7,
            label="Unobserved neighborhood nodes",
        ),
        Line2D(
            [0], [0],
            marker="o",
            color="w",
            markerfacecolor="gray",
            markersize=7,
            label="Observed in other labs",
        ),
        Line2D(
            [0], [0],
            marker="o",
            color="w",
            markerfacecolor="black",
            markeredgecolor="black",
            markersize=8,
            label="Highlighted lab",
        ),
    ]

    fig.legend(
        handles=legend_elements,
        loc="lower center",
        ncol=3,
        frameon=False,
    )

    fig.suptitle(
        f"Observed metabolites by institution in the Human1 local neighborhood subgraph (radius={radius})",
        fontsize=14,
    )

    plt.tight_layout(rect=[0, 0.05, 1, 0.95])
    plt.savefig(data_folder / "figs" / "lab_subgraph_regions_all.png", dpi=300, bbox_inches='tight')
    plt.show()

    return g, lab_nodes, pos

In [ ]:
g_labs, lab_nodes, pos = plot_each_lab_on_same_subgraph(
    graph=graph,
    lab_indices=lab_indices,
    graph_nodes=graph_nodes,
    radius=1,
    layout="spring",
    ncols=3,
    figsize=(16, 10),
)

In [ ]:
g_labs, lab_nodes, pos = plot_each_lab_on_same_subgraph(
    graph=graph,
    lab_indices=lab_indices,
    graph_nodes=graph_nodes,
    radius=2,
    layout="spring",
    ncols=3,
    figsize=(18, 12),
)

In [ ]:
pairwise_device_distances = {}
for device_name in devices.keys():
    pairwise_device_distances[device_name] = pairwise_distances(device_indices[device_name], distance_matrix)
    print(
        f"{device_name} mean: {pairwise_device_distances[device_name].mean():.4f}, std: {pairwise_device_distances[device_name].std():.4f}")

In [ ]:
plt.hist(
   distance_matrix.flatten(),
    bins=np.arange(1, distance_matrix.max()+2),
)

plt.xlabel("Shortest path distance")
plt.ylabel("Number of pairs")
plt.title("Pairwise distance distribution")
plt.show()

In [ ]:
def random_mean_distances(
    k,
    n_nodes,
    dist_matrix,
    n_samples=1000,
    seed=42,
):

    rng = np.random.default_rng(seed)

    means = []

    for _ in range(n_samples):

        sample = rng.choice(
            n_nodes,
            size=k,
            replace=False,
        )

        dists = pairwise_distances(
            sample,
            dist_matrix,
        )

        means.append(dists.mean())

    return np.array(means)

In [ ]:
random_means = random_mean_distances(
    k=len(device_indices['Plasma']),
    n_nodes=distance_matrix.shape[0],
    dist_matrix=distance_matrix,
)

In [ ]:
real_mean = distance_matrix.mean()

plt.hist(
    random_means,
    bins=50,
)

plt.axvline(
    real_mean,
    linewidth=3,
)

plt.xlabel("Mean pairwise distance")
plt.ylabel("Frequency")
plt.show()

In [ ]:
threshold = np.percentile(
    degrees,
    95
)

hub_nodes = np.where(
    degrees >= threshold
)[0]

In [ ]:
def distance_to_hub(
    device_nodes,
    hub_nodes,
    dist_matrix,
):

    return np.min(
        dist_matrix[
            np.ix_(
                device_nodes,
                hub_nodes
            )
        ],
        axis=1,
    )

In [ ]:
hub_distances = distance_to_hub(
    device_indices['Plasma'],
    hub_nodes,
    distance_matrix,
)

In [ ]:
plt.hist(
    hub_distances,
    bins=np.arange(
        hub_distances.max()+2
    )
)

plt.xlabel("Distance to nearest hub")
plt.ylabel("Count")
plt.show()

In [ ]:
def S0(idx,nodes):
    mask = np.zeros(len(nodes), dtype=bool)
    mask[idx] = True
    return mask

In [ ]:
S0(device_indices["Capitainer"],node_to_index.values())

In [ ]:
coverage.min_energy_anneling(S0(device_indices["Capitainer"],node_to_index.values()), mu=mu, gamma=gamma)[1]

In [ ]:
def gamma_annealing_sweep(device_name,nodes=node_to_index.values(), 
                          gamma_candidate=gamma, steps=15000, seed=42, 
                          Tmax=1.0, Tmin=1e-5, cooloing=0.995):
    
    device_mask = S0(device_indices[device_name], nodes)
    rows = []
    gamma_grid = np.linspace(max(0.01, 0.1 * gamma_candidate), max(0.01, 3.0 * gamma_candidate), 18)
    for gamma_candidate in gamma_grid:
        best_mask, best_energy, history = coverage.min_energy_anneling(
            S0=device_mask,
            mu=mu,
            gamma=float(gamma_candidate),
            Tmax=Tmax,
            Tmin=Tmin,
            cooloing=cooloing,
            seed=seed,
            steps=steps,
        )
        best_indices = np.where(best_mask == 1)[0]
        best_h, _, _ = H.compute(best_indices, mu=mu, gamma=float(gamma_candidate))
        rows.append(
            {
                "device": device_name,
                "k": int(device_mask.sum()),
                "gamma": float(gamma_candidate),
                "best_abs_H": float(best_energy),
                "best_signed_H": float(best_h),
                "history_min": float(np.min(history)) if len(history)>0 else float('nan'),
                "history_last": float(history[-1]) if len(history)>0 else float('nan'),
            }
        )
    return pd.DataFrame(rows)

In [ ]:
from matplotlib import pyplot as plt

In [ ]:
device_h_components = {}
for name in device_indices.keys():
    h_val, t1, t2 = H.compute(device_indices[name], mu=mu, gamma=gamma)
    device_h_components[name] = (h_val, t1, t2)

# Estimate a per-device gamma from the current H balance without creating the later energy arrays yet.
device_gamma_estimates = {}
for name, (h_val, t1, t2) in device_h_components.items():
    if t2 is None or abs(t2) < 1e-12:
        est = float(gamma)
    else:
        est = float((-t1 * float(gamma)) / float(t2))
    device_gamma_estimates[name] = max(1e-3, est)


def gamma_annealing_sweep(device_name, gamma_grid, steps=150000, seed=42, Tmax=1.0, Tmin=1e-5, cooloing=0.995):
    device_mask = S0(device_indices[device_name], node_to_index.values())
    rows = []

    for gamma_candidate in gamma_grid:
        best_mask, best_energy, history = coverage.min_energy_anneling(
            S0=device_mask,
            mu=mu,
            gamma=float(gamma_candidate),
            Tmax=Tmax,
            Tmin=Tmin,
            cooloing=cooloing,
            seed=seed,
            steps=steps,
        )
        best_indices = np.where(best_mask == 1)[0]
        best_h, _, _ = H.compute(best_indices, mu=mu, gamma=float(gamma_candidate))
        rows.append(
            {
                "device": device_name,
                "k": int(device_mask.sum()),
                "gamma": float(gamma_candidate),
                "best_abs_H": float(best_energy),
                "best_signed_H": float(best_h),
                "history_min": float(np.min(history)) if len(history) > 0 else float('nan'),
                "history_last": float(history[-1]) if len(history) > 0 else float('nan'),
            }
        )

    return pd.DataFrame(rows)

scan_frames = []
for name in device_indices.keys():
    gamma_estimate = device_gamma_estimates[name]
    gamma_grid = np.linspace(max(0.01, 0.1 * gamma_estimate), max(0.01, 3.0 * gamma_estimate), 18)
    scan_frames.append(gamma_annealing_sweep(name, gamma_grid, steps=1500, seed=42))

annealing_scan = pd.concat(scan_frames, ignore_index=True)

best_gamma_by_device = (
    annealing_scan.sort_values(["device", "best_abs_H"])
    .groupby("device", as_index=False)
    .first()[["device", "k", "gamma", "best_abs_H", "best_signed_H"]]
)

print(pd.DataFrame({"device": list(device_gamma_estimates.keys()),
                    "gamma_estimate": list(device_gamma_estimates.values())}).to_string(index=False))
print(best_gamma_by_device.to_string(index=False))

fig, ax = plt.subplots(figsize=(10, 6))
for device_name, device_df in annealing_scan.groupby("device"):
    ax.plot(device_df["gamma"], device_df["best_signed_H"], marker="o", linewidth=1.5, label=device_name)

ax.axhline(0.0, color="black", linewidth=1.0, linestyle="--")
ax.set_xlabel('s')
ax.set_ylabel('min E')
ax.set_title('Gamma sweep for device-specific k (gamma = s * gamma_estimate)')
ax.legend()
ax.grid(True, alpha=0.25)
out_dir = data_folder / 'figs'
plt.savefig(out_dir / 'gamma_sweep_plot.png', dpi=400, bbox_inches='tight')
plt.show()

In [ ]:
def sampling_energy_local(
    n,
    k,
    gamma,
    mu,
    reference_sample,
    swap_fraction=0.1,
    n_samples=10000,
    seed=42,
):
    """
    Sample configurations near a reference observation set.

    swap_fraction=0.1 means only 10% of nodes are replaced.
    """

    rng = np.random.default_rng(seed)

    energies = []
    samples = []

    reference_sample = np.asarray(reference_sample)

    n_swap = max(1, int(k * swap_fraction))

    all_nodes = np.arange(n)

    for _ in range(n_samples):

        sample = reference_sample.copy()

        remove_idx = rng.choice(k, size=n_swap, replace=False)

        remaining = np.setdiff1d(all_nodes, sample)

        new_nodes = rng.choice(
            remaining,
            size=n_swap,
            replace=False,
        )

        sample[remove_idx] = new_nodes

        E = H.compute(sample, gamma=gamma, mu=mu)[0]

        energies.append(abs(E))
        samples.append(sample)

    energies = np.asarray(energies)
    samples = np.asarray(samples)

    return (
        energies,
        samples[energies.argmin()],
        samples[energies.argmax()],
    )

Show relative min and maximun values of E

In [ ]:
H.distance_matrix[device_indices['Mitra'], :][:, device_indices['Mitra']]

Random swap sampling

In [ ]:
sampling_local_results = {}
for device_name in devices.keys():
    k = len(device_indices[device_name])
    reference_sample = device_indices[device_name]
    energies, best_sample, worst_sample = sampling_energy_local(
        n,
        k,
        gamma=0.22,
        mu=mu,
        reference_sample=reference_sample,
        swap_fraction=0.1,
        n_samples=90000,
        seed=42,
    )
    sampling_local_results[device_name] = (energies, best_sample, worst_sample)
    print(
        f"{device_name} (local): Best energy={energies.min():.4f}, Worst energy={energies.max():.4f}")

In [ ]:
def compute_robustness(
    reference_sample,
    n,
    gamma,
    mu,
    swap_fraction=0.1,
    n_samples=1000,
    seed=42,
):
    """
    Generate perturbed samples around a reference sample
    and compute the Hamiltonian distribution.

    Parameters
    ----------
    reference_sample : array-like
        List of observed node indices.

    n : int
        Total number of nodes in graph.

    gamma, mu : float
        Hamiltonian parameters.

    swap_fraction : float
        Fraction of metabolites replaced in each perturbation.

    n_samples : int
        Number of perturbations.

    seed : int
        Random seed.
    """

    rng = np.random.default_rng(seed)

    reference_sample = np.asarray(reference_sample)

    k = len(reference_sample)

    n_swap = max(1, int(k * swap_fraction))

    all_nodes = np.arange(n)

    energies = []

    for _ in range(n_samples):

        sample = reference_sample.copy()

        # choose nodes to remove
        remove_pos = rng.choice(
            k,
            size=n_swap,
            replace=False
        )

        # available replacement nodes
        available = np.setdiff1d(
            all_nodes,
            sample,
            assume_unique=False
        )

        replacement_nodes = rng.choice(
            available,
            size=n_swap,
            replace=False
        )

        sample[remove_pos] = replacement_nodes

        E = H.compute(
            sample,
            gamma=0.23,
            mu=mu
        )[0]

        energies.append(E)

    energies = np.asarray(energies)

    return {
        "mean": energies.mean(),
        "std": energies.std(),
        "min": energies.min(),
        "max": energies.max(),
        "energies": energies,
    }

In [ ]:
results = compute_robustness(
    reference_sample=device_indices["Capitainer"],
    n=n,
    gamma=0.23,
    mu=mu,
    swap_fraction=0.1,
    n_samples=5000,
)

print(results["mean"])
print(results["std"])

In [ ]:
plt.hist(results["energies"], bins=50)
plt.xlabel("Energy")
plt.ylabel("Count")
plt.show()

In [ ]:
z = (H.compute(device_indices["Capitainer"], gamma=0.23, mu=mu)[0]-results["mean"])/results["std"]
print(f"Z-score of observed configuration: {z:.2f}")

In [ ]:
E_random = sampling_results["Capitainer"][0]
E_real = H.compute(device_indices["Capitainer"], gamma=0.23, mu=mu)[0]

In [ ]:
p = np.mean(E_random <= E_real)
print(f"P-value of observed configuration compared to random sampling: {p:.4f}")

In [ ]:
device_energies = sampling_local_results['Capitainer'][0]
device_mean = np.mean(device_energies)
device_median = np.median(device_energies)
device_std = np.std(device_energies)
device_min = np.min(device_energies)
device_max = np.max(device_energies)

print(f"Device energy samples: n={len(device_energies)}, mean={device_mean:.6f}, median={device_median:.6f}, std={device_std:.6f}")

# Histogram and Gaussian overlay
counts, bins = np.histogram(device_energies, bins=80)
bin_width = bins[1] - bins[0]
bin_centers = (bins[:-1] + bins[1:]) / 2
max_count = counts.max() if len(counts)>0 else 1

plt.figure(figsize=(9,5))
plt.bar(bin_centers, counts, width=bin_width, color='lightgreen', edgecolor='none', alpha=0.9)

# Gaussian fit (scaled to histogram)
x = np.linspace(bins[0], bins[-1], 400)
pdf = (1.0 / (device_std * np.sqrt(2 * np.pi))) * np.exp(-0.5 * ((x - device_mean) / device_std) ** 2)
pdf_scaled = pdf * len(device_energies) * bin_width
plt.plot(x, pdf_scaled, color='k', linestyle='--', linewidth=2, label='Gaussian fit')

# Annotate summary stats
plt.axvline(device_mean, color='black', linestyle='-', linewidth=1.5, label=f'Mean = {device_mean:.4f}')

# Plot device energy for Capitainer (dotted line)
capitainer_val = abs(device_h_values['Capitainer'][0])
plt.axvline(capitainer_val, color='tab:orange', linestyle=':', linewidth=2.5, label=f"Capitainer energy = {capitainer_val:.4f}", zorder=9)

plt.xlabel('Energy')
plt.ylabel('Frequency')
plt.title('Energy distribution — Capitainer device')
plt.legend(loc='upper right', fontsize=9)
plt.grid(False)

out_dir = data_folder / 'figs'
plt.savefig(out_dir / 'energy_distribution_capitainer.png', dpi=400, bbox_inches='tight')

plt.show()

# Sampling random S0 and evaluate E on it
note: this is mean to be part of a library since is gonna be reuse in the future
but the ones I coded clearly have issues

### Energy distribution for variable k between the range of annotations

In [ ]:
variable_k_energy_samples, _, _ = coverage.sample_energy_variable_k(
    n=n, k_min=69, k_max=76, mu=mu, gamma=gamma, n_samples=100000, node_index=node_index)

### Energy distribution for Whole Blood

In [ ]:
energy_values_blood, min_mask_blood, max_mask_blood = sampling(
    node_index=node_index, k=70, mu=mu, gamma=0.01*gamma, n_samples=100000)

In [ ]:
max_mask_blood

In [ ]:
H.compute(device_indices['Blood'], mu=mu, gamma=gamma)

In [ ]:
graph_nodes

In [ ]:
np.unique(device_indices['Blood']).shape

In [ ]:
H.compute(max_mask_blood, mu=mu, gamma=gamma)

In [ ]:
blood_mean = np.mean(energy_values_blood)
blood_std = np.std(energy_values_blood)

print(f"Blood energy samples: n={len(energy_values_blood)}, mean={blood_mean:.6f}", 
      f"std={blood_std:.6f}")

# Histogram and Gaussian overlay
counts, bins = np.histogram(energy_values_blood, bins=80)
bin_width = bins[1] - bins[0]
bin_centers = (bins[:-1] + bins[1:]) / 2
max_count = counts.max() if len(counts)>0 else 1

plt.figure(figsize=(9,5))
plt.bar(bin_centers, counts, width=bin_width, color='lightcoral', edgecolor='none', alpha=0.9)

# Gaussian fit (scaled to histogram)
x = np.linspace(bins[0], bins[-1], 400)
pdf = (1.0 / (blood_std * np.sqrt(2 * np.pi))) * np.exp(-0.5 * ((x - blood_mean) / blood_std) ** 2)
pdf_scaled = pdf * len(energy_values_blood) * bin_width
plt.plot(x, pdf_scaled, color='k', linestyle='--', linewidth=2, label='Gaussian fit')

# Annotate summary stats
plt.axvline(blood_mean, color='black', linestyle='-', linewidth=1.5, label=f'Mean = {blood_mean:.4f}')

# Plot device energy for Blood (dotted line)
blood_val = device_energy_values['Blood']
plt.axvline(blood_val, color='tab:green', linestyle='--', linewidth=2.5,
             label=f"Blood energy = {blood_val:.4f}", zorder=9)

plt.xlabel('Energy')
plt.ylabel('Frequency')
plt.title('Energy distribution — Blood')
plt.legend(loc='upper right', fontsize=9)
plt.grid(False)

out_dir = data_folder / 'figs'
plt.savefig(out_dir / 'energy_distribution_blood.png', dpi=400, bbox_inches='tight')

plt.show()

### Energy distribution for Plasma

In [ ]:
energy_values_plasma, min_mask_plasma, max_mask_plasma = sampling(
    node_index=node_index, k=73, mu=mu, gamma=gamma, n_samples=80000
 )

### Energy distribution for Mitra

In [ ]:
energy_values_mitra, min_mask_mitra, max_mask_mitra = sampling(
    node_index=node_index, k=77, mu=mu, gamma=gamma, n_samples=100000 
 )

### Energy distribution for Capitainer

In [ ]:
energy_values_capitainer,min_mask_capitainer, max_mask_capitainer = sampling(
    node_index=node_index, k=75, mu=mu, gamma=gamma, n_samples=100000)

### Energy distribution intersection nodes

In [ ]:
energy_values_intersection, min_mask_intersection, max_mask_intersection = coverage.sample_energy(
    n=n, k=65, mu=mu, gamma=gamma, n_samples=80000
 )
h_values_intersection = energy_values_intersection.copy()
min_energy_intersection = np.where(min_mask_intersection == 1)[0]
max_energy_intersection = np.where(max_mask_intersection == 1)[0]

# 3- Visualization

### Combined Energy distribution for all devices
- Fro variable $k$ between the range of the observed nodes

In [ ]:
# Use sampled distribution from variable-k sampling
sampled_distribution = variable_k_energy_samples

# Compute histogram bins and counts for precise control
counts, bins = np.histogram(sampled_distribution, bins=100)
bin_width = bins[1] - bins[0]
bin_centers = (bins[:-1] + bins[1:]) / 2

plt.figure(figsize=(10, 6))
# Light-blue bars with no contour
plt.bar(bin_centers, counts, width=bin_width, color='lightblue', edgecolor='none', align='center', alpha=0.9)

# Gaussian approximation (scaled to histogram counts)
sampled_mean = np.mean(sampled_distribution)
sampled_std = np.std(sampled_distribution)
x = np.linspace(bins[0], bins[-1], 500)
pdf = (1.0 / (sampled_std * np.sqrt(2 * np.pi))) * np.exp(-0.5 * ((x - sampled_mean) / sampled_std) ** 2)
pdf_scaled = pdf * len(sampled_distribution) * bin_width
plt.plot(x, pdf_scaled, color='k', linestyle='--', linewidth=2, label='Gaussian fit')

# Colors for the five devices
colors = {
    'Capitainer': 'blue',
    'Mitra': 'black',
    'Whatman': 'green',
    'Blood': 'red',
    'Plasma': 'purple'
}

# Plot vertical lines for each device energy; emphasize Blood and Plasma
for name, color in colors.items():
    val = round(device_energy_values[name])
    plt.axvline(val, color=color, linestyle='--', linewidth=2, label=f'{name} = {val:.0f}', zorder=8)

plt.xlabel('Energy')
plt.ylabel('Frequency')
plt.title('Energy Distribution with Device Energies and Gaussian Approximation')
plt.legend(loc='upper left', fontsize=9)
plt.grid(False)
plt.savefig(f'{data_folder}/figs/energy_distribution_with_devices.png', dpi=400, bbox_inches='tight')
plt.show()

In [ ]:
device_energy_values['Mitra']

### Whatman device

### Whole Blood

In [ ]:
mitra_energy_samples = energy_values_mitra
mitra_mean = np.mean(mitra_energy_samples)
mitra_median = np.median(mitra_energy_samples)
mitra_std = np.std(mitra_energy_samples)
mitra_min = np.min(mitra_energy_samples)
mitra_max = np.max(mitra_energy_samples)

print(f"Mitra energy samples: n={len(mitra_energy_samples)}, mean={mitra_mean:.6f}, median={mitra_median:.6f}, std={mitra_std:.6f}")

# Histogram and Gaussian overlay
counts, bins = np.histogram(mitra_energy_samples, bins=80)
bin_width = bins[1] - bins[0]
bin_centers = (bins[:-1] + bins[1:]) / 2
max_count = counts.max() if len(counts)>0 else 1

plt.figure(figsize=(9,5))
plt.bar(bin_centers, counts, width=bin_width, color='orange', edgecolor='none', alpha=0.9)

# Gaussian fit (scaled to histogram)
x = np.linspace(bins[0], bins[-1], 400)
pdf = (1.0 / (mitra_std * np.sqrt(2 * np.pi))) * np.exp(-0.5 * ((x - mitra_mean) / mitra_std) ** 2)
pdf_scaled = pdf * len(mitra_energy_samples) * bin_width
plt.plot(x, pdf_scaled, color='k', linestyle='--', linewidth=2, label='Gaussian fit')

# Annotate summary stats
plt.axvline(mitra_mean, color='black', linestyle='-', linewidth=1.5, label=f'Mean = {mitra_mean:.4f}')

# Plot device energy for Mitra (dotted line)
mitra_val = device_energy_values['Mitra']
plt.axvline(mitra_val, color='tab:blue', linestyle=':', linewidth=2.5, label=f"Mitra energy = {mitra_val:.4f}", zorder=9)

plt.xlabel('Energy')
plt.ylabel('Frequency')
plt.title('Energy distribution — Mitra')
plt.legend(loc='upper right', fontsize=9)
plt.grid(False)

out_dir = data_folder / 'figs'
plt.savefig(out_dir / 'energy_distribution_mitra.png', dpi=400, bbox_inches='tight')

plt.show()

In [ ]:
plasma_energy_samples = energy_values_plasma
plasma_mean = np.mean(plasma_energy_samples)
plasma_median = np.median(plasma_energy_samples)
plasma_std = np.std(plasma_energy_samples)
plasma_min = np.min(plasma_energy_samples)
plasma_max = np.max(plasma_energy_samples)

print(f"Plasma energy samples: n={len(plasma_energy_samples)}, mean={plasma_mean:.6f}, median={plasma_median:.6f}, std={plasma_std:.6f}")

# Histogram and Gaussian overlay
counts, bins = np.histogram(plasma_energy_samples, bins=80)
bin_width = bins[1] - bins[0]
bin_centers = (bins[:-1] + bins[1:]) / 2
max_count = counts.max() if len(counts)>0 else 1

plt.figure(figsize=(9,5))
plt.bar(bin_centers, counts, width=bin_width, color='gray', edgecolor='none', alpha=0.9)

# Gaussian fit (scaled to histogram)
x = np.linspace(bins[0], bins[-1], 400)
pdf = (1.0 / (plasma_std * np.sqrt(2 * np.pi))) * np.exp(-0.5 * ((x - plasma_mean) / plasma_std) ** 2)
pdf_scaled = pdf * len(plasma_energy_samples) * bin_width
plt.plot(x, pdf_scaled, color='k', linestyle='--', linewidth=2, label='Gaussian fit')

# Annotate summary stats
plt.axvline(plasma_mean, color='black', linestyle='-', linewidth=1.5, label=f'Mean = {plasma_mean:.4f}')

# Plot device energy for Plasma (dotted line)
plasma_val = device_energy_values['Plasma']
plt.axvline(
    plasma_val, color='tab:orange', linestyle=':', 
    linewidth=2.5, label=f"Plasma energy = {plasma_val:.4f}", zorder=9)

plt.xlabel('Energy')
plt.ylabel('Frequency')
plt.title('Energy distribution — Plasma')
plt.legend(loc='upper right', fontsize=9)
plt.grid(False)

out_dir = data_folder / 'figs'
plt.savefig(out_dir / 'energy_distribution_plasma.png', dpi=400, bbox_inches='tight')

plt.show()

In [ ]:
capitainer_energy_samples = energy_values_capitainer
capitainer_mean = np.mean(capitainer_energy_samples)
capitainer_median = np.median(capitainer_energy_samples)
capitainer_std = np.std(capitainer_energy_samples)
capitainer_min = np.min(capitainer_energy_samples)
capitainer_max = np.max(capitainer_energy_samples)

print(f"Capitainer energy samples: n={len(capitainer_energy_samples)}, mean={capitainer_mean:.6f}, median={capitainer_median:.6f}, std={capitainer_std:.6f}")

# Histogram and Gaussian overlay
counts, bins = np.histogram(capitainer_energy_samples, bins=80)
bin_width = bins[1] - bins[0]
bin_centers = (bins[:-1] + bins[1:]) / 2
max_count = counts.max() if len(counts)>0 else 1

plt.figure(figsize=(9,5))
plt.bar(bin_centers, counts, width=bin_width, color='plum', edgecolor='none', alpha=0.9)

# Gaussian fit (scaled to histogram)
x = np.linspace(bins[0], bins[-1], 400)
pdf = (1.0 / (capitainer_std * np.sqrt(2 * np.pi))) * np.exp(-0.5 * ((x - capitainer_mean) / capitainer_std) ** 2)
pdf_scaled = pdf * len(capitainer_energy_samples) * bin_width
plt.plot(x, pdf_scaled, color='k', linestyle='--', linewidth=2, label='Gaussian fit')

# Annotate summary stats
plt.axvline(capitainer_mean, color='black', linestyle='-', linewidth=1.5, label=f'Mean = {capitainer_mean:.4f}')

# Plot device energy for Capitainer (dotted line)
capitainer_val = device_energy_values['Capitainer']
plt.axvline(
    capitainer_val, color='tab:green', linestyle=':', 
    linewidth=2.5, label=f"Capitainer energy = {capitainer_val:.4f}", zorder=9)

plt.xlabel('Energy')
plt.ylabel('Frequency')
plt.title('Energy distribution — Capitainer')
plt.legend(loc='upper right', fontsize=9)
plt.grid(False)

out_dir = data_folder / 'figs'
plt.savefig(out_dir / 'energy_distribution_capitainer.png', dpi=400, bbox_inches='tight')

plt.show()

In [ ]:
import importlib.util

phase_diagrams_path = project_root / "examples" / "etc_utils" / "phase_diagrams.py"
spec = importlib.util.spec_from_file_location("phase_diagrams", phase_diagrams_path)
phase_diagrams_module = importlib.util.module_from_spec(spec)
assert spec is not None and spec.loader is not None
spec.loader.exec_module(phase_diagrams_module)
phase_diagram_values = phase_diagrams_module.phase_diagram_values

def plot_phase_diagram_for_graph(
    G,
    *,
    mu=1.0,
    gamma=1.0,
    kmax=20,
    scale_max=12,
    scale_steps=1,
    k_steps=1,
    cmap="magma",
    contour_color="white",
    title="Phase Diagram",
    show_transitions=True,
    transition_levels=8,
    highlight_scale=None,
    highlight_color="red",
    highlight_linewidth=2.0,
):
    H_obj = Hamiltonian(G)

    k_values, ratio, H = phase_diagram_values(
        Hamiltonian=H_obj,
        mu=float(mu),
        gamma=float(gamma),
        kmax=int(kmax),
        scale_max=float(scale_max),
        scale_steps=float(scale_steps),
        k_steps=int(k_steps),
    )

    # k_values, ratio, H are now numpy arrays
    # H shape: (len(k_values), len(scale_values))
    # ratio shape: (len(k_values), len(scale_values))

    scale_values = np.arange(0.25, scale_max + scale_steps, scale_steps)

    # Compute extent: [left, right, bottom, top]
    scale_min = scale_values[0]
    scale_max_val = scale_values[-1]
    k_min = k_values[0]
    k_max = k_values[-1]
    extent = [scale_min, scale_max_val, k_min, k_max]

    fig, ax = plt.subplots(figsize=(9, 5), constrained_layout=True)
    im = ax.imshow(
        H,
        aspect="auto",
        origin="lower",
        interpolation="bilinear",
        cmap=cmap,
        extent=extent,
    )

    # Add transition boundaries (contour lines)
    if show_transitions:
        h_min, h_max = np.nanmin(H), np.nanmax(H)
        contour_levels = np.linspace(h_min, h_max, transition_levels)
        cs = ax.contour(
            scale_values,
            k_values,
            H,
            levels=contour_levels,
            colors=contour_color,
            linewidths=0.7,
            alpha=0.5,
        )
        ax.clabel(cs, inline=True, fontsize=7, fmt="%.3f")

    # Add vertical line at specific mu/gamma value
    if highlight_scale is not None:
        ax.axvline(
            x=float(highlight_scale),
            color=highlight_color,
            linestyle="--",
            linewidth=highlight_linewidth,
            alpha=0.8,
            label=f"s = {highlight_scale:.3f}",
        )
        ax.legend(loc="upper right")

    cbar = fig.colorbar(im, ax=ax)
    cbar.set_label(r"$H \approx 0$")
    ax.set_xlabel(r"$s$")
    ax.set_ylabel(r"$k$")
    ax.set_title(title)

    print(f"Graph: n={G.number_of_nodes()}, m={G.number_of_edges()}")
    print("Computed phase diagram.")
    if show_transitions:
        print(f"Transition boundaries shown at {transition_levels} levels.")
    if highlight_scale is not None:
        print(f"Highlighted transition line at μ/γ = {highlight_scale:.3f}")

    figures_dir = project_root / "examples/figures"
    figures_dir.mkdir(parents=True, exist_ok=True)

    filename = f"PD_{title}.png"
    plt.savefig(figures_dir / filename, dpi=300)
    plt.show()

    return fig, ax, (k_values, ratio, H)

In [ ]:
intersection_energy_samples = energy_values_intersection
intersection_mean = np.mean(intersection_energy_samples)
intersection_median = np.median(intersection_energy_samples)
intersection_std = np.std(intersection_energy_samples)
intersection_min = np.min(intersection_energy_samples)
intersection_max = np.max(intersection_energy_samples)

print(f"Intersection energy samples: n={len(intersection_energy_samples)}, mean={intersection_mean:.6f}, median={intersection_median:.6f}, std={intersection_std:.6f}")

# Histogram and Gaussian overlay
counts, bins = np.histogram(intersection_energy_samples, bins=80)
bin_width = bins[1] - bins[0]
bin_centers = (bins[:-1] + bins[1:]) / 2
max_count = counts.max() if len(counts)>0 else 1

plt.figure(figsize=(9,5))
plt.bar(bin_centers, counts, width=bin_width, color='lightblue', edgecolor='none', alpha=0.9)

# Gaussian fit (scaled to histogram)
x = np.linspace(bins[0], bins[-1], 400)
pdf = (1.0 / (intersection_std * np.sqrt(2 * np.pi))) * np.exp(-0.5 * ((x - intersection_mean) / intersection_std) ** 2)
pdf_scaled = pdf * len(intersection_energy_samples) * bin_width
plt.plot(x, pdf_scaled, color='k', linestyle='--', linewidth=2, label='Gaussian fit')

# Annotate summary stats
plt.axvline(intersection_mean, color='black', linestyle='-', linewidth=1.5, label=f'Mean = {intersection_mean:.4f}')

# Plot a reference value from the sampled minimum-energy subset
reference_intersection_energy = H.energy(min_energy_intersection, mu=mu, gamma=gamma)
plt.axvline(
    reference_intersection_energy, color='tab:red', linestyle=':', 
    linewidth=2.5, label=f"Minimum sampled intersection energy = {reference_intersection_energy:.4f}", zorder=9)

plt.xlabel('Energy')
plt.ylabel('Frequency')
plt.title('Energy distribution — Intersection')
plt.legend(loc='upper right', fontsize=9)
plt.grid(False)

out_dir = data_folder / 'figs'
plt.savefig(out_dir / 'energy_distribution_intersection.png', dpi=400, bbox_inches='tight')

plt.show()